In [1]:
print('Generate gif for retraction and deformation ')

Generate gif for retraction and deformation 


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch
from PIL import Image
import io, os

OUT = os.getcwd()
os.makedirs(OUT, exist_ok=True)

# ── helpers ──────────────────────────────────────────────────────────────────
BG   = "#0f1117"
BLUE = "#4a9eff"
TEAL = "#1dc87c"
ORANGE = "#e86b3d"
GRAY = "#8a8fa8"
WHITE = "#f0f2ff"

def fig_to_pil(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    buf.seek(0)
    return Image.open(buf).copy()

# ═══════════════════════════════════════════════════════════════════════════════
# GIF 1 – RETRACTION  (annulus → inner circle via r(x) = x/|x|)
# Each frame: arrows grow from outer rim inward, points slide along them
# ═══════════════════════════════════════════════════════════════════════════════
N_FRAMES = 40
frames1 = []

# sample points on the annulus (r between 0.55 and 1)
np.random.seed(42)
n_pts = 26
angles = np.linspace(0, 2*np.pi, n_pts, endpoint=False)
radii  = np.random.uniform(0.55, 1.0, n_pts)
pts_x  = radii * np.cos(angles)
pts_y  = radii * np.sin(angles)
# retraction target on inner circle
tgt_x  = 0.5 * np.cos(angles)
tgt_y  = 0.5 * np.sin(angles)

for fi in range(N_FRAMES):
    t = fi / (N_FRAMES - 1)          # 0 → 1
    fig, ax = plt.subplots(figsize=(5, 5), facecolor=BG)
    ax.set_facecolor(BG)
    ax.set_aspect("equal")
    ax.set_xlim(-1.45, 1.45)
    ax.set_ylim(-1.45, 1.45)
    ax.axis("off")

    # annulus fill (ring)
    outer = plt.Circle((0,0), 1.0, color=BLUE, alpha=0.13, zorder=1)
    hole  = plt.Circle((0,0), 0.5, color=BG,   zorder=2)
    ax.add_patch(outer)
    ax.add_patch(hole)

    # outer boundary
    outer_ring = plt.Circle((0,0), 1.0, color=BLUE, fill=False,
                             linewidth=1.5, alpha=0.5, zorder=3)
    ax.add_patch(outer_ring)

    # inner circle A (retract) — always vivid
    inner_ring = plt.Circle((0,0), 0.5, color=ORANGE, fill=False,
                             linewidth=2.5, zorder=5)
    ax.add_patch(inner_ring)

    # moving points
    cx = pts_x + t * (tgt_x - pts_x)
    cy = pts_y + t * (tgt_y - pts_y)
    ax.scatter(cx, cy, s=28, color=TEAL, zorder=6, alpha=0.9)

    # faint trail lines
    for i in range(n_pts):
        ax.plot([pts_x[i], cx[i]], [pts_y[i], cy[i]],
                color=TEAL, lw=0.6, alpha=0.25, zorder=4)

    # labels (only first & last frames look cluttered otherwise)
    ax.text(0, -1.35, "Retraction  r(x) = x / |x|",
            ha="center", va="center", color=WHITE,
            fontsize=10, fontweight="bold",
            fontfamily="DejaVu Sans")
    ax.text(0, -1.22, f"t = {t:.2f}",
            ha="center", va="center", color=GRAY, fontsize=9)
    # legend dots
    ax.scatter([-1.3], [1.3], s=30, color=TEAL, zorder=10)
    ax.text(-1.18, 1.3, "points in X", color=TEAL, fontsize=8, va="center")
    ax.plot([-1.32, -1.15], [1.12, 1.12], color=ORANGE, lw=2.5)
    ax.text(-1.04, 1.12, "A = inner circle", color=ORANGE, fontsize=8, va="center")

    frames1.append(fig_to_pil(fig))
    plt.close(fig)

# hold on last frame
for _ in range(15):
    frames1.append(frames1[-1].copy())

frames1[0].save(
    f"{OUT}/retraction.gif",
    save_all=True, append_images=frames1[1:],
    duration=60, loop=0, optimize=False
)
print("retraction.gif saved")

# ═══════════════════════════════════════════════════════════════════════════════
# GIF 2 – DEFORMATION RETRACTION  (disk → center via H(x,t) = (1-t)·x)
# ═══════════════════════════════════════════════════════════════════════════════
N2 = 48
frames2 = []

# sample points uniformly inside disk
np.random.seed(7)
n2 = 34
r2   = np.sqrt(np.random.uniform(0.04, 1.0, n2))
a2   = np.random.uniform(0, 2*np.pi, n2)
px2  = r2 * np.cos(a2)
py2  = r2 * np.sin(a2)

for fi in range(N2):
    t = fi / (N2 - 1)
    fig, ax = plt.subplots(figsize=(5, 5), facecolor=BG)
    ax.set_facecolor(BG)
    ax.set_aspect("equal")
    ax.set_xlim(-1.45, 1.45)
    ax.set_ylim(-1.45, 1.45)
    ax.axis("off")

    # disk fill — shrinks with t to give "collapsing" feel
    disk_alpha = 0.15 * (1 - 0.6*t)
    disk = plt.Circle((0,0), 1.0, color=TEAL, alpha=disk_alpha, zorder=1)
    ax.add_patch(disk)

    # disk boundary
    disk_ring = plt.Circle((0,0), 1.0, color=TEAL, fill=False,
                            linewidth=1.5, alpha=max(0.1, 0.5*(1-t)), zorder=3)
    ax.add_patch(disk_ring)

    # H(x,t) = (1-t)*x
    scale = 1 - t
    cx2 = px2 * scale
    cy2 = py2 * scale

    # trail lines from original position to current
    for i in range(n2):
        ax.plot([px2[i], cx2[i]], [py2[i], cy2[i]],
                color=TEAL, lw=0.5, alpha=0.2, zorder=4)

    # points
    ax.scatter(cx2, cy2, s=22, color=BLUE, zorder=6, alpha=0.9)

    # fixed center point A = {0}
    ax.scatter([0], [0], s=90, color=ORANGE, zorder=8)
    ax.scatter([0], [0], s=25, color=WHITE, zorder=9)

    ax.text(0, -1.35, "Deformation retraction  H(x, t) = (1−t)·x",
            ha="center", va="center", color=WHITE,
            fontsize=9.5, fontweight="bold", fontfamily="DejaVu Sans")
    ax.text(0, -1.22, f"t = {t:.2f}",
            ha="center", va="center", color=GRAY, fontsize=9)

    # legend
    ax.scatter([-1.3], [1.3], s=28, color=BLUE, zorder=10)
    ax.text(-1.18, 1.3, "points in X = D²", color=BLUE, fontsize=8, va="center")
    ax.scatter([-1.3], [1.12], s=70, color=ORANGE, zorder=10)
    ax.scatter([-1.3], [1.12], s=20, color=WHITE,  zorder=11)
    ax.text(-1.18, 1.12, "A = {center}", color=ORANGE, fontsize=8, va="center")

    frames2.append(fig_to_pil(fig))
    plt.close(fig)

# hold last frame
for _ in range(18):
    frames2.append(frames2[-1].copy())

frames2[0].save(
    f"{OUT}/deformation_retraction.gif",
    save_all=True, append_images=frames2[1:],
    duration=55, loop=0, optimize=False
)
print("deformation_retraction.gif saved")
print("Done!")

retraction.gif saved
deformation_retraction.gif saved
Done!
